# 🏦 Screening Credit Agentic AI — Ringkasan Dashboard Streamlit

Dashboard produksinya ada di `app.py` + `pages/*.py` (Streamlit multi-page app, dijalankan lewat `streamlit run app.py`). Streamlit punya server & event loop sendiri sehingga **tidak bisa dijalankan langsung di dalam notebook** — jadi notebook ini bukan salinan dashboard yang "jalan", melainkan:

1. **Ringkasan alur** — struktur halaman, arsitektur 7-agent pipeline, dan bagaimana semuanya terhubung.
2. **Reproduksi logic inti secara live** — `utils/agent_pipeline.py` murni Python/pandas/numpy (tidak bergantung pada Streamlit), jadi scoring-nya dijalankan ulang di sini agar hasil & visualisasinya bisa diverifikasi langsung dari notebook, bukan cuma dideskripsikan.

Untuk melihat dashboard interaktifnya sendiri, jalankan dari terminal di root project:
```bash
streamlit run app.py
```

## Struktur Proyek Dashboard

```
app.py                              # Halaman utama (Overview)
pages/
  1_Daftar_Pengajuan.py             # Tabel semua pengajuan + filter & pencarian
  2_Detail_Nasabah.py               # Detail 1 nasabah: 6 kartu agent + kartu keputusan risk agent
  3_Simulasi.py                     # Form input manual → jalankan pipeline live (tanpa LLM)
  4_Monitoring_Portofolio.py        # Breakdown zona per cabang/industri, heatmap, ranking cabang
utils/
  agent_pipeline.py                 # Inti logic: 7-agent rule-based scoring
  data_loader.py                    # Load master_dataset.csv, cache, scoring batch (@st.cache_data)
  ui_components.py                  # Komponen render kartu agent (dipakai bersama oleh 2 halaman)
data/processed/master_dataset.csv   # Dataset pengajuan kredit UMKM (input dashboard)
```

Semua halaman memanggil `load_master_data()` (yang men-cache dataset yang sudah discoring) atau `score_application()` untuk input manual — jadi satu logic scoring dipakai konsisten di seluruh dashboard, tidak ada duplikasi rule antar halaman.

## Alur 7-Agent Scoring Pipeline

Setiap pengajuan (baik dari dataset maupun form Simulasi) melewati 7 agent rule-based berikut — **tidak ada pemanggilan LLM/API eksternal**, semuanya deterministik:

**Gate (hard-reject) — kalau salah satu gagal, langsung "Tidak Layak" tanpa peduli skor lain:**
1. **Identity Agent** — validasi NIK (16 digit) & usia pemilik (21–65 tahun).
2. **Credit History Agent** — baca kolektibilitas SLIK terburuk; status *Macet* = hard reject.
3. **DHN Agent** — cek Daftar Hitam Nasional; terdaftar = hard reject.

**Scoring (prinsip 5C) — dibobot lalu digabung jadi `combined_score`:**
4. **Collateral Agent** (bobot 20%) — rasio nilai agunan vs pinjaman + kecocokan kepemilikan.
5. **Financial Agent** (bobot 30%) — pertumbuhan omset YoY + margin laba bersih.
6. **Cashflow Agent** (bobot 20%) — rata-rata saldo rekening vs omset, penalti overdraft & rekening dormant.
   *(Credit History Agent di atas juga dipakai sebagai skor Character dengan bobot 30% di tahap ini.)*

**Orkestrator:**
7. **Risk Agent** — menggabungkan ke-4 skor berbobot menjadi `combined_score`, memetakannya ke salah satu dari 4 tier keputusan (`Layak` / `Layak Bersyarat` / `Perlu Review Ulang` / `Tidak Layak`), lalu merekomendasikan jenis kredit (KMK/KI), nominal disetujui, tenor, dan bunga.

`utils/agent_pipeline.py:score_application()` menjalankan ke-7 agent untuk satu baris data; `score_dataframe()` memanggilnya berulang untuk seluruh dataset (dipakai oleh `data_loader.load_master_data()` saat dashboard start).

In [1]:
import sys
from pathlib import Path

# Notebook ini ada di notebooks/, sedangkan utils/ ada di root project
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.agent_pipeline import score_application, score_dataframe, COLLECT_LABEL_MAP
from utils.data_loader import load_master_data

print("Project root:", PROJECT_ROOT)

2026-08-24 08:06:23.315 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


Project root: C:\Users\ASUS\Documents\4. Bootcamp BNI\Capstone Project DA


In [2]:
# Load & scoring dataset — persis fungsi yang dipanggil app.py & semua halaman
# di pages/ saat dashboard start (hasilnya di-cache oleh @st.cache_data di sana).
df = load_master_data()

print(f"{len(df):,} pengajuan dimuat & discoring oleh 7-agent pipeline".replace(",", "."))
df[["application_id", "company_name", "branch_name", "industry", "risk_score", "decision", "zone"]].head(10)

2026-08-24 08:06:23.341 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-08-24 08:06:23.343 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-08-24 08:06:24.173 
  command:

    streamlit run C:\ProgramData\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]


2026-08-24 08:06:24.175 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-08-24 08:06:24.177 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


...

2026-08-24 08:06:26.779 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-08-24 08:06:26.782 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-08-24 08:06:26.784 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


3.000 pengajuan dimuat & discoring oleh 7-agent pipeline


,application_id,company_name,branch_name,industry,risk_score,decision,zone
0,APP202600001,UD Santoso Abadi,KCP Bogor Baranangsiang,Manufaktur,0.761,Layak,Hijau
1,APP202600002,UD Wijaya Mandiri,KCP Bekasi Barat,Jasa,0.614,Layak Bersyarat,Kuning
2,APP202600003,UD Kusuma Sejahtera,KCP Cibubur,Transportasi,0.529,Perlu Review Ulang,Kuning
3,APP202600004,UD Susanto Makmur,KCP Bogor Baranangsiang,Perdagangan,0.655,Layak Bersyarat,Kuning
4,APP202600005,CV Wijaya Sejahtera,KCP Bogor Baranangsiang,Transportasi,0.881,Layak,Hijau
5,APP202600006,UD Suryadi Sejahtera,KCP Tangerang BSD,Jasa,0.854,Layak,Hijau
6,APP202600007,PT Panjaitan Mandiri,KCP Pluit,Perdagangan,0.575,Perlu Review Ulang,Kuning
7,APP202600008,UD Kurniawan Jaya,KCP Tangerang BSD,Jasa,0.519,Perlu Review Ulang,Kuning
8,APP202600009,UD Wijaya Mandiri,KCP Cibubur,Transportasi,0.585,Perlu Review Ulang,Kuning
9,APP202600010,CV Rahman Jaya,KCP Pluit,Perdagangan,0.629,Layak Bersyarat,Kuning


## Contoh Scoring Manual (setara halaman "Simulasi")

Halaman `pages/3_Simulasi.py` menyusun dict dari input form lalu memanggil `score_application()` yang sama persis dipakai untuk seluruh dataset di atas — jadi hasil form Simulasi selalu konsisten dengan hasil batch. Berikut contoh reproduksinya di notebook, dengan satu profil nasabah simulatif:

In [3]:
contoh_pengajuan = {
    "NIK": "3276010601750001",
    "owner_age": 35,
    "slik_worst_collectability": 1,          # 1 = Lancar (lihat COLLECT_LABEL_MAP)
    "slik_n_banks": 1,
    "status_dhn": "Tidak",
    "collateral_ratio": 1.5,                  # nilai agunan 1.5x nominal pinjaman
    "ownership_match": "Ya",
    "revenue_growth_pct": 0.10,               # +10% YoY
    "profit_margin_2025": 0.11,               # 11%
    "bank_best_avg_balance_6m": 1_000_000,
    "monthly_turnover_est": 2_000_000,
    "bank_total_overdraft_6m": 0,
    "bank_any_dormant": False,
    "loan_requested": 100_000_000,
    "collateral_liquidation_value": 400_000_000,
}

hasil = score_application(contoh_pengajuan)
for agent_key, res in hasil.items():
    if agent_key == "risk":
        continue
    print(f"{agent_key:15s} | score={res['score']:.2f} | status={res['status']}")

risk = hasil["risk"]
print("\n--- Risk Agent (orkestrator) ---")
print(f"Skor gabungan : {risk['combined_score']:.2f}")
print(f"Keputusan     : {risk['decision']} (zona {risk['zone']})")
print(f"Insight       : {risk['insight']}")

identity        | score=1.00 | status=Valid
credit_history  | score=1.00 | status=Lancar
dhn             | score=1.00 | status=Bersih
collateral      | score=1.00 | status=Baik
financial       | score=0.58 | status=Cukup
cashflow        | score=0.62 | status=Baik

--- Risk Agent (orkestrator) ---
Skor gabungan : 0.80
Keputusan     : Layak (zona Hijau)
Insight       : Skor gabungan 0.80 → keputusan Layak (zona Hijau).


## Alur 5 Halaman Dashboard

| Halaman | Isi | Interaksi |
|---|---|---|
| **Overview** (`app.py`) | KPI (total pengajuan, approval rate, rata-rata risk score, nominal disetujui), distribusi zona risiko, validasi hard rule DHN, fairness check approval rate per gender, komposisi nasabah, ringkasan per cabang | Filter: cabang, industri, rentang tanggal |
| **Daftar Pengajuan** | Tabel seluruh pengajuan dengan kolom hasil scoring, warna baris mengikuti zona/keputusan | Filter sidebar (cabang, industri, keputusan, zona, range risk score, pencarian nama/ID) + tombol lihat detail |
| **Detail Nasabah** | 6 kartu agent (skor, status, catatan) + 1 kartu Risk Agent (keputusan akhir, jenis kredit, nominal, tenor, bunga) untuk satu nasabah terpilih | Pilih ID pengajuan (atau lanjut dari Daftar Pengajuan) |
| **Simulasi** | Form input manual lengkap → jalankan pipeline live → tampilkan kartu hasil yang sama seperti Detail Nasabah | Isi form lalu submit — tidak menyentuh dataset asli |
| **Monitoring Portofolio** | Breakdown zona risiko per cabang & per industri (stacked bar), heatmap risk score cabang×industri, ranking approval rate per cabang | Tidak ada filter — tampilan agregat seluruh portofolio |

Dua halaman (**Detail Nasabah** & **Simulasi**) berbagi komponen render yang sama dari `utils/ui_components.py` (`render_full_result`), jadi tampilan kartu agent konsisten di keduanya walau sumber datanya beda (baris dataset vs input form).

Berikut reproduksi dua visual kunci dari **Overview** dan **Monitoring Portofolio** langsung dari `df` yang sudah discoring di atas:

In [4]:
import plotly.express as px

ZONE_COLORS = {"Hijau": "#16a34a", "Kuning": "#d97706", "Merah": "#dc2626"}
ZONE_ORDER = ["Hijau", "Kuning", "Merah"]

# --- Sama seperti panel "Distribusi Zona Risiko" di halaman Overview ---
zone_counts = df["zone"].value_counts().reindex(ZONE_ORDER).fillna(0).reset_index()
zone_counts.columns = ["zone", "count"]
fig = px.bar(zone_counts, x="zone", y="count", color="zone", color_discrete_map=ZONE_COLORS,
             text="count", title="Distribusi Zona Risiko — Seluruh Portofolio")
fig.update_layout(showlegend=False, xaxis_title=None, yaxis_title="Jumlah Pengajuan")
fig.show()

# --- Validasi hard rule DHN — sama seperti panel di Overview ---
dhn_yes = df[df["status_dhn"] == "Ya"]
pct_rejected = (dhn_yes["decision"] == "Tidak Layak").mean() * 100 if len(dhn_yes) else 100.0
print(f"Nasabah DHN 'Ya' otomatis Tidak Layak: {pct_rejected:.1f}% (idealnya 100%)")

Nasabah DHN 'Ya' otomatis Tidak Layak: 100.0% (idealnya 100%)


In [5]:
# --- Sama seperti panel "Ranking Approval Rate per Cabang" di Monitoring Portofolio ---
branch_summary = (
    df.groupby("branch_name")
    .agg(
        total_pengajuan=("application_id", "count"),
        approval_rate=("decision", lambda s: s.isin(["Layak", "Layak Bersyarat"]).mean() * 100),
        avg_risk_score=("risk_score", "mean"),
    )
    .reset_index()
    .sort_values("approval_rate", ascending=False)
)
branch_summary["approval_rate"] = branch_summary["approval_rate"].round(1)
branch_summary["avg_risk_score"] = branch_summary["avg_risk_score"].round(2)

fig2 = px.bar(
    branch_summary.sort_values("approval_rate"), x="approval_rate", y="branch_name", orientation="h",
    text_auto=".1f", color="approval_rate", color_continuous_scale=["#dc2626", "#d97706", "#16a34a"],
    title="Approval Rate per Cabang",
)
fig2.update_layout(yaxis_title=None, xaxis_title="Approval Rate (%)", coloraxis_showscale=False)
fig2.show()

branch_summary

,branch_name,total_pengajuan,approval_rate,avg_risk_score
5,KCP Kelapa Gading,314,84.4,0.73
1,KCP Bogor Baranangsiang,272,82.7,0.72
9,KCP Tebet,282,82.6,0.73
6,KCP Kemang,312,82.1,0.72
3,KCP Cikini,293,81.9,0.72
0,KCP Bekasi Barat,309,81.9,0.72
7,KCP Pluit,315,81.3,0.72
8,KCP Tangerang BSD,279,80.6,0.72
2,KCP Cibubur,330,79.4,0.72
4,KCP Depok Margonda,294,79.3,0.71


## Menjalankan Dashboard Interaktif Penuh

Angka & chart di atas sudah identik dengan yang dihasilkan dashboard, karena keduanya memanggil fungsi yang sama persis di `utils/`. Yang tidak bisa direproduksi di notebook adalah bagian interaktif Streamlit-nya sendiri (filter widget, sidebar, navigasi multi-page, form Simulasi). Untuk itu, jalankan dari terminal di root project:

```bash
pip install -r requirements.txt
streamlit run app.py
```

Dashboard akan terbuka di browser (`localhost:8501`) dengan 5 halaman seperti dijelaskan di atas.